In [ ]:
from glob import glob
import os
import pandas as pd
import json
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import shutil
import base64
import numpy as np

import cv2
from ultralytics import YOLO

from openai import OpenAI
from tqdm import tqdm


client = OpenAI(api_key=KEY, base_url="https://openrouter.ai/api/v1")
MODEL = "openai/gpt-5-mini"
MODEL_NAME = "GPT Mini"

In [2]:
# qwen/qwen3.7-plus

# mistralai/mistral-small-2603
# mistralai/mistral-medium-3.1
# mistralai/mistral-large-2512

# openai/gpt-5-mini
# openai/gpt-4.1

# anthropic/claude-opus-4.8
# anthropic/claude-sonnet-4.6

In [3]:
from openai import OpenAI
import cv2
import base64
import numpy as np


def query_mistral_ocr_htr(
    client: OpenAI,
    picture: np.ndarray,
    system_prompt: str,
    user_prompt:str,
    model: str,
) -> str:
    """
    Query a Mistral model via OpenRouter for Handwritten Text Recognition (HTR).

    Args:
        client: OpenAI-compatible client pointed at OpenRouter
        picture: Image in cv2/numpy format (BGR or grayscale)
        system_prompt: System prompt to guide the OCR/HTR behavior
        model: OpenRouter model string

    Returns:
        Extracted text from the handwritten image
    """
    # Encode the cv2 image to base64
    success, buffer = cv2.imencode(".jpg", picture)
    if not success:
        raise ValueError("Failed to encode image to JPEG format")

    image_base64 = base64.b64encode(buffer.tobytes()).decode("utf-8")

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{image_base64}"
                        },
                    },
                    {
                        "type": "text",
                        "text": user_prompt,
                    },
                ],
            },
        ],
    )

    return response.choices[0].message.content

In [4]:
system_prompt = """
You are an expert paleographer specializing in early 20th century French cursive handwriting (1890–1940). You have deep knowledge of the vocabulary, expressions, formulaic phrases, and orthography typical of French personal correspondence of this period (postcards, letters).

Your task is verbatim Handwritten Text Recognition (HTR).

When a word or character is ambiguous or partially legible:
- Use all available context to determine the most probable reading: the surrounding words, the sentence grammar, the topic of the correspondence, and common epistolary formulas of the era.
- Use your knowledge of how specific letters are formed in this script style (French cursive, early 20th c.) to disambiguate visually similar shapes.
- Always commit to a single best reading. Never leave blanks or uncertainty markers.

Rules:
- Transcribe every letter exactly as you interpret it — do not modernize spelling or correct errors made by the writer.
- Preserve original line breaks, punctuation, and capitalization.
- Transcribe all text zones in reading order: address block first, then message body, then signature.
- Return only the transcribed text. No commentary, no explanations.
                """

user_prompt="""
                You are an expert paleographer, specifically trained on early XXth century french handwritten text recognition. 
                You have to transcript the text present on the following picture exactly as it is. 
                You will return a text string containing the whole text transcribed.
                Don't invent anything. Just transcript what you see. 
              """

In [21]:
files_path = "./ToKeep/**/*.tif"
output_folder = f"./transcriptions"
os.makedirs(output_folder, exist_ok=True)
for file_path in tqdm(glob(files_path)):
    output_path_elements = file_path.split(sep="/")[-2:]
    subfolder = output_path_elements[0]
    basename = output_path_elements[1].split(sep=".")[0]
    basename = f"{basename}"
    sub_output_folder = os.path.join(output_folder, subfolder)
    output_path = os.path.join(sub_output_folder, f"{basename}.txt")
    os.makedirs(sub_output_folder, exist_ok=True)
    
    img = cv2.imread(file_path)
    
    answer_gpt = query_mistral_ocr_htr(client=client, picture=img, system_prompt=system_prompt, user_prompt=user_prompt, model=MODEL)
    with open(output_path, "w") as output_file:
        output_file.write(answer_gpt)

100%|██████████| 45/45 [32:45<00:00, 43.68s/it]


In [16]:
print(answer_gpt)

Cercle des Capucines
6. Boulevard des Capucines

Paris 20 Août

Mon cher ami
Ci contre la liste des membres
qui font partie de la
commission qui choisit le
candidat à Galignani.
Je ne t'en certifie pas
l'exactitude absolue
Il y a au Secrétariat de
l'Institut un aimable
homme spécialement
chargé des dossiers
un Monsieur Bonnaire

qui ne peux te refuser de
te renseigner pour tout
ce qui concerne mon cas
Merci de tout ce que tu
tenteras pour moi en ces
pénibles circonstances
et bonne poignée de main

J. Neymart.
18 rue des Gatines

Pr. Ch. Widor
Laloux
Chabas
E. de Rothschild
Lefebvre sculp
Coutan sculp
J. L. Sulpis graveur
Waltener graveur
Je crois pouvoir compter sur la 1re colonne

Paladilhe musicien
Ch. Dubois musicien
Delaplane architecte
Allar sculp.
Michel prof. Collège de France


In [29]:
print(file_path)

./ALLEMAGNE/FO-A-1647_2.tif
